In [7]:
import pandas as pd
from micom.workflows import load_results
from skbio.diversity import beta_diversity
from itertools import product
import numpy as np

In [9]:
res_simulated = load_results('../goll_et.al_2020_IBS_FMT/models_simulated/res_western_simulated.zip')
res_actual = load_results('../goll_et.al_2020_IBS_FMT/models/res_western.zip')

DonorRecipientMapping = pd.read_csv('../goll_et.al_2020_IBS_FMT/raw/DonorRecipientMapping.csv')[['recipient', 'donor', 'clinical_response']]
DonorRecipientMapping['donor'] = DonorRecipientMapping['donor'].str.split('_').str[0]
DonorRecipientMapping['recipient'] = DonorRecipientMapping['recipient'].str.split('_').str[0].str.replace('-0', '')
DonorRecipientMapping['comparison'] = DonorRecipientMapping.apply(lambda x: x['donor'] + '_vs_' + x['recipient'] + '-0', axis=1)

abundance_actual = (
    res_actual.growth_rates[['sample_id', 'taxon', 'abundance']]
    .query('sample_id.str.contains("D") | sample_id.str.endswith("-6")')
    .assign(
        sample_id=lambda x: np.where(
            ~x['sample_id'].str.startswith('D'),
            x['sample_id'].str.replace('-6', ''),
            x['sample_id']
        )
    )
    .merge(
        DonorRecipientMapping[['recipient', 'donor']],
        left_on='sample_id',
        right_on='recipient',
        how='left'
    )
    .assign(
        comparison=lambda x: np.where(
            x['recipient'].isna(),
            x['sample_id'],  # Donor samples
            x['donor'] + '_vs_' + x['recipient'] + '-0'  # Recipient samples
        )
    )
    .pivot_table(index='taxon', columns='comparison', values='abundance', fill_value=0.0)
)

abundance_actual_donors = abundance_actual.loc[:, ~abundance_actual.columns.str.endswith('-0')]
abundance_actual_recipients = abundance_actual.loc[:, abundance_actual.columns.str.endswith('-0')]

abundance_calculated = (
    res_simulated.growth_rates[['sample_id', 'taxon', 'abundance']]
    .query('~sample_id.isin(@abundance_actual_recipients.columns)')
    .pivot_table(index='taxon', columns='sample_id', values='abundance', fill_value=0.0)
)

abundance_all = pd.concat([abundance_actual_donors, abundance_actual_recipients, abundance_calculated], axis=1).fillna(0)
b = beta_diversity(metric='braycurtis', counts=abundance_all.T.values)
distMatrix = pd.DataFrame(b.data, index=abundance_all.T.index, columns=abundance_all.T.index)
donors = abundance_actual_donors.columns.tolist()
T6Recipients = list(set(abundance_all.columns) - set(abundance_actual_donors.columns))
bray_curtis_dissimilarity = {
    'score': [],
    'donor': [],
    'recipient': []
}

for pairs in product(donors, T6Recipients):
    donor_1 = pairs[0]
    donor_2, recipient = pairs[1].split('_vs_')[0], pairs[1].split('_vs_')[1]
    recipient = recipient.replace('-0', '')
    if donor_1 == donor_2:
        score = distMatrix.loc[pairs[0], pairs[1]]
        bray_curtis_dissimilarity['score'].append(score)
        bray_curtis_dissimilarity['donor'].append(donor_1)
        bray_curtis_dissimilarity['recipient'].append(recipient)

bray_curtis_dissimilarity_table = (
    pd.DataFrame(bray_curtis_dissimilarity)
    .pivot_table(
        index='donor',
        columns='recipient',
        values='score',
        fill_value=0.0
    )
    .transpose()
)

bray_curtis_dissimilarity = (
    pd.DataFrame(bray_curtis_dissimilarity)
    .merge(DonorRecipientMapping[['recipient', 'clinical_response']], on=['recipient'], how='left')
)
bray_curtis_dissimilarity['clinical_response'] = pd.Categorical(bray_curtis_dissimilarity['clinical_response'], ['responder', 'non-responder'], ordered=True)
bray_curtis_dissimilarity_table.to_csv('./data/3C.csv')

/opt/anaconda3/envs/qiime2/lib/python3.9/site-packages/micom/workflows/results.py:55: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  tab = pd.read_csv(zippy.open(f"{attr}.csv", "r"))
/opt/anaconda3/envs/qiime2/lib/python3.9/site-packages/micom/workflows/results.py:55: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  tab = pd.read_csv(zippy.open(f"{attr}.csv", "r"))
